### Классический генетический алгоритм. Оптимизация гиперпараметров при помощи PyGad

In [1]:
import pandas as pd
import pygad
from tqdm import tqdm
import time

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split

### KNeighbors Classifier

Создадим 20 поколений, в каждом из которых будет 10 особей

In [2]:
num_generations = 20
sol_per_pop = 10
total_evals = num_generations * sol_per_pop

gene_space = [
    {'low': 1, 'high': 50},  # n_neighbours
    {'low': 1, 'high': 10}     # p
]

Функция, определяющая оценку текущей особи

In [4]:
def fitness_func(ga_instance, solution, solution_idx):
    global pbar
    pbar.update(1)

    model = KNeighborsClassifier(
        n_neighbors=int(solution[0]),
        p=int(solution[1]),
        weights='distance',
        n_jobs=-1
    )

    score = cross_val_score(
        model,
        x_train,
        y_train,
        cv=5,
        scoring='f1_macro',
        n_jobs=-1
    ).mean()

    return score

Настройка алгоритма PyGad. Используем `tqdm` для просмотра изменения популяции

In [5]:
n_samples = [100, 500, 1000, 3000]
m_features = [5, 8, 11]

results = []

In [ ]:
for n in n_samples:
    for m in m_features:
        data = pd.read_csv(f"../classical_ml_methods/f1_data/f1_data_{n}_s_{m}_f.csv")

        pbar = tqdm(total=total_evals, desc=f"knn pygad optimization n={n} m={m}")

        y = data['collision']
        x = data.drop(['collision'], axis=1)

        global x_train, y_train
        x_train, x_test, y_train, y_test = train_test_split(
            x, y, test_size=0.2, random_state=81
            )
        
        ga_instance = pygad.GA(
            num_generations=num_generations,
            sol_per_pop=sol_per_pop,
            num_parents_mating=5,
            num_genes=2,
            fitness_func=fitness_func,
            gene_space=gene_space,
            parent_selection_type="rank",
            keep_parents=2,
            mutation_percent_genes=50,
            random_seed=81
        )

        start_time = time.time()
        ga_instance.run()
        search_time = time.time() - start_time

        solution, solution_fitness, _ = ga_instance.best_solution()
        
        results.append({
            'samples (n)': n,
            'features (m)': m,
            'best_n_neighbors': int(solution[0]),
            'best_p': int(solution[1]),
            'f1-score': solution_fitness,
            'search_time (sec)': search_time
        })

KNN GA n=3000 m=8:  98%|█████████▊| 195/200 [00:06<00:00, 28.01it/s]


In [ ]:
results_df = pd.DataFrame(results)
results_df

,samples (n),features (m),best_n_neighbors,best_p,f1-score,search_time (sec)
0,100,5,1,1,1.000000,7.712762
1,100,8,23,2,0.989474,4.194678
2,100,11,1,1,0.889976,4.892298
3,500,5,10,2,0.992775,4.962584
4,500,8,18,2,0.980857,4.945957
5,500,11,1,1,0.888899,4.978802
6,1000,5,14,7,0.996363,4.980714
7,1000,8,12,1,0.982750,5.154177
8,1000,11,2,2,0.877795,5.538127
9,3000,5,2,6,0.996209,6.064537
